In [1]:
import numpy as np
from qutip import *

In [5]:
def get_Sn(spin, N, n):
    dims = int(2*spin + 1)
    sp = spin_Jp(spin)
    sp_n = tensor([qeye(dims)] * n + [sp] + [qeye(dims)]*(N-n-1))
    return sp_n

def get_coupled_basis_state(spin, num_spins, J, M, d):
    dims = int(2 * spin + 1)
    combo = get_all_coupled_spin_combos(num_spins, J)[d]
    state = np.zeros((dims ** num_spins, 1))
    total_dims = dims ** num_spins
    for i in range(total_dims):
        ms_inds = 2 * np.array(convert_num_to_spin_str(i, [spin] * num_spins)) - 0.5
        state[total_dims - i - 1] += multi_clebsch([spin] * num_spins, ms_inds, combo, M)
    return Qobj(state, dims=[[dims] * num_spins, [1] * num_spins])

def get_all_coupled_spin_combos(num_spins, coupled_spin_number):
    valid_combos = []
    coupled_number_bounds = []
    bound = 1
    for i in range(num_spins - 1):
        coupled_number_bounds.append(bound)
        bound += 0.5
    curr_str = [0] * (num_spins - 1)
    while curr_str != -1:
        is_valid = curr_str[-1] == coupled_spin_number + 0.5 or curr_str[-1] == coupled_spin_number - 0.5
        counter = 2
        while is_valid and counter <= len(curr_str):
            is_valid = curr_str[-counter] == curr_str[-counter + 1] + 0.5 or curr_str[-counter] == curr_str[
                -counter + 1] - 0.5
            counter += 1
        if is_valid:
            valid_combos.append(curr_str[1:] + [coupled_spin_number])
        curr_str = iterate_spin_numbers(curr_str, coupled_number_bounds)
    return valid_combos


def iterate_spin_numbers(spin_numbers, coupled_number_bounds):
    max_val = convert_spin_str_to_num(coupled_number_bounds, coupled_number_bounds)
    curr_num = convert_spin_str_to_num(spin_numbers, coupled_number_bounds) + 1
    if curr_num > max_val:
        return -1
    else:
        return convert_num_to_spin_str(curr_num, coupled_number_bounds)


def convert_spin_str_to_num(spin_numbers, coupled_number_bounds):
    num_spins = len(spin_numbers)
    curr_num = 0
    curr_dims = 1
    for i in range(num_spins):
        curr_num += int(2 * spin_numbers[num_spins - 1 - i]) * curr_dims
        curr_dims *= int(2 * coupled_number_bounds[num_spins - 1 - i] + 1)
    return curr_num


def convert_num_to_spin_str(num, coupled_number_bounds):
    dim_list = np.array(2 * np.array(coupled_number_bounds) + 1, dtype=int)
    curr_dims = np.prod(dim_list) // dim_list[0]
    ind_list = [num // curr_dims]
    curr_num = num - (curr_dims * ind_list[0])
    ind_list[0] /= 2
    for k in range(1, len(dim_list)):
        curr_dims //= dim_list[k]
        ind_list.append(curr_num // curr_dims)
        curr_num -= ind_list[k] * curr_dims
        ind_list[k] /= 2
    return ind_list

def to_decimal(combo):
    first_one_pos = 0
    not_found = combo[first_one_pos] == 0
    while not_found:
        first_one_pos += 1
        not_found = combo[first_one_pos] == 0
    max_power = len(combo) - first_one_pos - 1
    val = np.longlong(0)
    for i in range(len(combo) - first_one_pos):
        val += combo[i + first_one_pos] * (2 ** (max_power - i))
    return val


def to_binary(dec_val, string_len=None):
    power_list = []
    val_total = 0
    while val_total != dec_val:
        curr_power = 0
        curr_val = dec_val - val_total
        while curr_val > 1:
            curr_val //= 2
            curr_power += 1
        power_list.append(curr_power)
        val_total += 2 ** curr_power
    if val_total == 0:
        bin_list = [0]
    else:
        bin_list = [0] * (power_list[0] + 1)
    for power in power_list:
        bin_list[power] = 1
    if string_len is not None and string_len > len(bin_list):
        bin_list = bin_list + [0] * (string_len - len(bin_list))
    return bin_list[::-1]


def iterate_combo(combo):
    num_ones = 0
    counter = 0
    for digit in combo:
        if digit == 1:
            num_ones += 1
    dec_combo = to_decimal(combo)
    not_found = True
    while not_found:
        dec_combo += np.longlong(1)
        bin_combo = to_binary(dec_combo)
        counter = 0
        for digit in bin_combo:
            if digit == 1:
                counter += 1
        not_found = counter != num_ones
    new_combo = to_binary(dec_combo)
    if len(new_combo) > len(combo):
        return [-1]
    else:
        new_combo = to_binary(dec_combo, string_len=len(combo))
        return new_combo


def multi_clebsch(uncoupled_spin_numbers, uncoupled_ms_numbers, coupled_spin_numbers, coupled_M):
    if len(coupled_spin_numbers) == 1:
        return clebsch(uncoupled_spin_numbers[0], uncoupled_spin_numbers[1], coupled_spin_numbers[0],
                       uncoupled_ms_numbers[0], uncoupled_ms_numbers[1], coupled_M)
    else:
        multi_cleb = 0
        dims = int(2 * coupled_spin_numbers[-2] + 1)
        for i in range(dims):
            ms_curr = i - coupled_spin_numbers[-2]
            multi_cleb += multi_clebsch(uncoupled_spin_numbers[:-1], uncoupled_ms_numbers[:-1],
                                        coupled_spin_numbers[:-1], ms_curr) * clebsch(coupled_spin_numbers[-2],
                                                                                      uncoupled_spin_numbers[-1],
                                                                                      coupled_spin_numbers[-1], ms_curr,
                                                                                      uncoupled_ms_numbers[-1],
                                                                                      coupled_M)
        return multi_cleb

def get_bitstrings_from_states(state, num_spins):
    bitstrings = []
    coeffs = []
    for i in range(2**num_spins):
        if not np.isclose(state[i], 0):
            bitstrings.append(to_binary(i, string_len=num_spins))
            coeffs.append(state[i])
    return coeffs, bitstrings

### In this notebook, we will be finding all $O_{J,d}$ for all $J<NS$ where $N$ is the number of spins and $S=1/2$. $d$ is the degenercy index as defined in the paper

### Case 1: $N=3$

First find all eigenvalues $J_2$ of $(\vec{S}_1 + \vec{S}_2)^2$ such that $J_2 \pm 1/2 = J_3 = 1/2$. This can be easily solved for without numerics as $J_2 = 0$ and $J_2 = 1$. In general however, this can be done with the function get_all_coupled_spin_combos as follows

In [3]:
N=3
J_3 = 0.5
degeneracy_list = get_all_coupled_spin_combos(N, J_3)
print(degeneracy_list)

[[0.0, 0.5], [1.0, 0.5]]


As expected, there are two sequences of $(J_2, J_3)$. This means that $d=1,2$ and that $d=1$ corresponds to the sequence $(J_2, J_3) = (0,1/2)$ and $d=2$ to $(1,1/2)$. Next we must determine the $O_{J,d}$. We set $J=1/2$, $d=0,1$, $M=-1/2$ and spin=0.5 in thae arguments of the function get_coupled_basis_states. As explained in the paper, the coefficents of the state $|J,K\rangle_d$ in the uncoupled basis give the coefficients of the operators $O_{J,d}$ since $O_{J,d}|\downarrow\rangle^{\otimes N} = |J,K\rangle_d$

In [7]:
spin = 1/2
s1 = get_Sn(spin, N, 0)
s2 = get_Sn(spin, N, 1)
s3 = get_Sn(spin, N, 2)

Get bitstrings returns the coupled basis state here with $N=3$ spins, $J_3 = 1/2$, $K=-1/2$, and $d=1$. get_bitstrings_from_states then gets the decimal indices from get_coupled_basis_state and converts them to binary where $|0\rangle = |\uparrow\rangle$ and $|1\rangle = |\downarrow\rangle$. The array coeffs are the prefactors in front of each uncoupled basis state.

In [8]:
#Note that this function will only work for spin=0.5 . The argument is somewhat pointless
coeffs, bitstrings = get_bitstrings_from_states(get_coupled_basis_state(spin, N, 0.5, -0.5, 0), N)
print(coeffs)
print(bitstrings)

[array([0.70710678+0.j]), array([-0.70710678+0.j])]
[[0, 1, 1], [1, 0, 1]]


Now we can construct the operator $O_{1/2,1}$. Wherever there is a zero in the $n$th place of the list bitstrings, put an operator $S^+_n$. Finally, multiply each term by its corresponding composite Clebsch-Gordan coefficient. We will construct $O_{1/2,1}$ and verify properties 1 and 2 from the paper.

In [11]:
O_half_1 = (s1 - s2)/np.sqrt(2)
# property 1
print(O_half_1 * Qobj(get_coupled_basis_state(spin, N, 1.5, 0.5, 0), dims=[[int(2*spin+1)]*N , [1]*N]))
# property 2
print(O_half_1 * Qobj(get_coupled_basis_state(spin, N, 0.5, 0.5, 0), dims=[[int(2*spin+1)]*N , [1]*N]))

Quantum object: dims=[[2, 2, 2], [1]], shape=(8, 1), type='ket', dtype=Dense
Qobj data =
[[0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]]
Quantum object: dims=[[2, 2, 2], [1]], shape=(8, 1), type='ket', dtype=Dense
Qobj data =
[[-1.]
 [ 0.]
 [ 0.]
 [ 0.]
 [ 0.]
 [ 0.]
 [ 0.]
 [ 0.]]


Now for $d=2$

In [12]:
#Note that this function will only work for spin=0.5 . The argument is somewhat pointless
coeffs, bitstrings = get_bitstrings_from_states(get_coupled_basis_state(spin, N, 0.5, -0.5, 1), N)
print(coeffs)
print(bitstrings)

[array([0.40824829+0.j]), array([0.40824829+0.j]), array([-0.81649658+0.j])]
[[0, 1, 1], [1, 0, 1], [1, 1, 0]]


In [14]:
O_half_2 = (s1 + s2 - 2*s3)/np.sqrt(6)
# property 1
print(O_half_2 * Qobj(get_coupled_basis_state(spin, N, 1.5, 0.5, 0), dims=[[int(2*spin+1)]*N , [1]*N]))
# property 2
print(O_half_2 * Qobj(get_coupled_basis_state(spin, N, 0.5, 0.5, 1), dims=[[int(2*spin+1)]*N , [1]*N]))

Quantum object: dims=[[2, 2, 2], [1]], shape=(8, 1), type='ket', dtype=Dense
Qobj data =
[[5.55111512e-17]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]]
Quantum object: dims=[[2, 2, 2], [1]], shape=(8, 1), type='ket', dtype=Dense
Qobj data =
[[-1.]
 [ 0.]
 [ 0.]
 [ 0.]
 [ 0.]
 [ 0.]
 [ 0.]
 [ 0.]]


### N = 4

In [15]:
N=4
s1 = get_Sn(spin, N, 0)
s2 = get_Sn(spin, N, 1)
s3 = get_Sn(spin, N, 2)
s4 = get_Sn(spin, N, 3)

Find the $d$ corresponding to $J_4 = 0$

In [16]:
J_4 = 0
degeneracy_list = get_all_coupled_spin_combos(N, J_4)
print(degeneracy_list)

[[0.0, 0.5, 0], [1.0, 0.5, 0]]


$O_{0, 1}$

In [17]:
coeffs, bitstrings = get_bitstrings_from_states(get_coupled_basis_state(spin, N, 0, 0, 0), N)
print(coeffs)
print(bitstrings)

[array([0.5+0.j]), array([-0.5+0.j]), array([-0.5+0.j]), array([0.5+0.j])]
[[0, 1, 0, 1], [0, 1, 1, 0], [1, 0, 0, 1], [1, 0, 1, 0]]


In [20]:
O_01 = 0.5*(s1*s3 - s1*s4 - s2*s3 + s2*s4)
print(O_01 * get_coupled_basis_state(spin, N, 2, 0, 0))
print(O_01 * get_coupled_basis_state(spin, N, 0, 0, 0))

Quantum object: dims=[[2, 2, 2, 2], [1]], shape=(16, 1), type='ket', dtype=Dense
Qobj data =
[[0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]]
Quantum object: dims=[[2, 2, 2, 2], [1]], shape=(16, 1), type='ket', dtype=Dense
Qobj data =
[[1.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]]


In [21]:
coeffs, bitstrings = get_bitstrings_from_states(get_coupled_basis_state(spin, N, 0, 0, 1), N)
print(coeffs)
print(bitstrings)

[array([0.57735027+0.j]), array([-0.28867513+0.j]), array([-0.28867513+0.j]), array([-0.28867513+0.j]), array([-0.28867513+0.j]), array([0.57735027+0.j])]
[[0, 0, 1, 1], [0, 1, 0, 1], [0, 1, 1, 0], [1, 0, 0, 1], [1, 0, 1, 0], [1, 1, 0, 0]]


In [23]:
O_02 = (2*s1*s2 - s1*s3 - s1*s4 - s2*s3 - s2*s4 + 2*s3*s4)/np.sqrt(12)
print(O_02 * get_coupled_basis_state(spin, N, 2, 0, 0))
print(O_02 * get_coupled_basis_state(spin, N, 0, 0, 1))

Quantum object: dims=[[2, 2, 2, 2], [1]], shape=(16, 1), type='ket', dtype=Dense
Qobj data =
[[-5.55111512e-17]
 [ 0.00000000e+00]
 [ 0.00000000e+00]
 [ 0.00000000e+00]
 [ 0.00000000e+00]
 [ 0.00000000e+00]
 [ 0.00000000e+00]
 [ 0.00000000e+00]
 [ 0.00000000e+00]
 [ 0.00000000e+00]
 [ 0.00000000e+00]
 [ 0.00000000e+00]
 [ 0.00000000e+00]
 [ 0.00000000e+00]
 [ 0.00000000e+00]
 [ 0.00000000e+00]]
Quantum object: dims=[[2, 2, 2, 2], [1]], shape=(16, 1), type='ket', dtype=Dense
Qobj data =
[[1.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]]


For $J_4 = 1$, $d=1,2,3$

In [30]:
J_4 = 1
degeneracy_list = get_all_coupled_spin_combos(N, J_4)
print(degeneracy_list)

[[0.0, 0.5, 1], [1.0, 0.5, 1], [1.0, 1.5, 1]]


In [24]:
coeffs, bitstrings = get_bitstrings_from_states(get_coupled_basis_state(spin, N, 1, 0, 0), N)
print(coeffs)
print(bitstrings)

[array([0.5+0.j]), array([0.5+0.j]), array([-0.5+0.j]), array([-0.5+0.j])]
[[0, 1, 0, 1], [0, 1, 1, 0], [1, 0, 0, 1], [1, 0, 1, 0]]


In [25]:
O_11 = 0.5 * (s1*s3 + s1*s4 - s2*s3 - s2*s4)
print(O_11 * get_coupled_basis_state(spin, N, 2, 0, 0))
print(O_11 * get_coupled_basis_state(spin, N, 1, 0, 0))

Quantum object: dims=[[2, 2, 2, 2], [1]], shape=(16, 1), type='ket', dtype=Dense
Qobj data =
[[0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]]
Quantum object: dims=[[2, 2, 2, 2], [1]], shape=(16, 1), type='ket', dtype=Dense
Qobj data =
[[-1.]
 [ 0.]
 [ 0.]
 [ 0.]
 [ 0.]
 [ 0.]
 [ 0.]
 [ 0.]
 [ 0.]
 [ 0.]
 [ 0.]
 [ 0.]
 [ 0.]
 [ 0.]
 [ 0.]
 [ 0.]]


In [26]:
coeffs, bitstrings = get_bitstrings_from_states(get_coupled_basis_state(spin, N, 1, 0, 1), N)
print(coeffs)
print(bitstrings)

[array([0.57735027+0.j]), array([-0.28867513+0.j]), array([0.28867513+0.j]), array([-0.28867513+0.j]), array([0.28867513+0.j]), array([-0.57735027+0.j])]
[[0, 0, 1, 1], [0, 1, 0, 1], [0, 1, 1, 0], [1, 0, 0, 1], [1, 0, 1, 0], [1, 1, 0, 0]]


In [27]:
O_12 = (2*s1*s2 - s1*s3 + s1*s4 - s2*s3 + s2*s4 - 2*s3*s4)/np.sqrt(12)
print(O_12 * get_coupled_basis_state(spin, N, 2, 0, 0))
print(O_12 * get_coupled_basis_state(spin, N, 1, 0, 1))

Quantum object: dims=[[2, 2, 2, 2], [1]], shape=(16, 1), type='ket', dtype=Dense
Qobj data =
[[0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]]
Quantum object: dims=[[2, 2, 2, 2], [1]], shape=(16, 1), type='ket', dtype=Dense
Qobj data =
[[-1.]
 [ 0.]
 [ 0.]
 [ 0.]
 [ 0.]
 [ 0.]
 [ 0.]
 [ 0.]
 [ 0.]
 [ 0.]
 [ 0.]
 [ 0.]
 [ 0.]
 [ 0.]
 [ 0.]
 [ 0.]]


In [28]:
coeffs, bitstrings = get_bitstrings_from_states(get_coupled_basis_state(spin, N, 1, 0, 2), N)
print(coeffs)
print(bitstrings)

[array([0.40824829+0.j]), array([0.40824829+0.j]), array([-0.40824829+0.j]), array([0.40824829+0.j]), array([-0.40824829+0.j]), array([-0.40824829+0.j])]
[[0, 0, 1, 1], [0, 1, 0, 1], [0, 1, 1, 0], [1, 0, 0, 1], [1, 0, 1, 0], [1, 1, 0, 0]]


In [29]:
O_13 = (s1*s2 + s1*s3 - s1*s4 + s2*s3 - s2*s4 - s3*s4)/np.sqrt(6)
print(O_13 * get_coupled_basis_state(spin, N, 2, 0, 0))
print(O_13 * get_coupled_basis_state(spin, N, 1, 0, 2))

Quantum object: dims=[[2, 2, 2, 2], [1]], shape=(16, 1), type='ket', dtype=Dense
Qobj data =
[[2.77555756e-17]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]]
Quantum object: dims=[[2, 2, 2, 2], [1]], shape=(16, 1), type='ket', dtype=Dense
Qobj data =
[[-1.]
 [ 0.]
 [ 0.]
 [ 0.]
 [ 0.]
 [ 0.]
 [ 0.]
 [ 0.]
 [ 0.]
 [ 0.]
 [ 0.]
 [ 0.]
 [ 0.]
 [ 0.]
 [ 0.]
 [ 0.]]


For $N>4$, simply repeat the procedure above.

In [35]:
N=5
s1 = get_Sn(spin, N, 0)
s2 = get_Sn(spin, N, 1)
s3 = get_Sn(spin, N, 2)
s4 = get_Sn(spin, N, 3)
s5 = get_Sn(spin, N, 4)

In [36]:
J_5 = 0.5
degeneracy_list = get_all_coupled_spin_combos(N, J_5)
print(degeneracy_list)

[[0.0, 0.5, 0.0, 0.5], [0.0, 0.5, 1.0, 0.5], [1.0, 0.5, 0.0, 0.5], [1.0, 0.5, 1.0, 0.5], [1.0, 1.5, 1.0, 0.5]]


In [33]:
coeffs, bitstrings = get_bitstrings_from_states(get_coupled_basis_state(spin, N, 0.5, -0.5, 0), N)
print(coeffs)
print(bitstrings)

[array([0.5+0.j]), array([-0.5+0.j]), array([-0.5+0.j]), array([0.5+0.j])]
[[0, 1, 0, 1, 1], [0, 1, 1, 0, 1], [1, 0, 0, 1, 1], [1, 0, 1, 0, 1]]


Do the rest yourself.